# EEG_15 — DHSLP (Li et al. 2025): implementazione fedele

**DOI**: 10.1088/1741-2552/adeec8 — J. Neural Eng. 22 (2025) 046030

| Aspetto | Paper | EEG_15 | |
|---------|-------|--------|---|
| Canali / classi | 64ch / 5cl | 61ch / 4cl | ⚠️ dataset |
| Feature dim | 128D | 122D | ⚠️ dataset |
| Split 300/50/50 | stratificato (60/cl, 10/cl, 10/cl) | stratificato (75/cl, ~12/cl, ~12/cl) | ✓ |
| f grid | {10…90} 9 val | {10…90} 9 val | ✓ |
| N_CONFIGS | 3087 | 3087 | ✓ |
| Algorithm 1, grid ξ/α/β, MAX_ITER=30, 1-NN test, PCC per-soggetto | ✓ | ✓ | ✓ |

**Deviazioni inevitabili**: solo canali (61 vs 64) e classi (4 vs 5) — dataset diverso.

In [ ]:
import json, logging, re, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from scipy import signal as scipy_signal
from scipy.linalg import eigh
from scipy.stats import pearsonr
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg15')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents)
                     if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg15'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS     = 61
N_SAMPLES      = 384
SFREQ          = 256
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'

# Feature (Li et al. §4.1)
GAMMA_LOW      = 30
GAMMA_HIGH     = 100
N_SELECT_FEAT  = 2     # top-2 temporal features per PCC → 2×61 = 122D

# Split identico al paper §4.1
N_LABELED      = 300
N_UNLABELED    = 50
N_TEST         = 50
RANDOM_SEED    = 42

# Grid search identico al paper §4.2
XI_LIST    = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
ALPHA_LIST = [2**k for k in (-6, -4, -2, 0, 2, 4, 6)]
BETA_LIST  = [2**k for k in (-6, -4, -2, 0, 2, 4, 6)]
F_DIM_LIST = [10, 20, 30, 40, 50, 60, 70, 80, 90]   # identico al paper §4.2
MAX_ITER   = 30

N_CONFIGS = len(XI_LIST) * len(ALPHA_LIST) * len(BETA_LIST) * len(F_DIM_LIST)
log.info(f'N_CONFIGS = {N_CONFIGS}  (7×7×7×9 = 3087)')

DATA_METRIC   = 'abs_pcc'
WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess[int(m.group(1))][int(m.group(2))].append(p)

ALL_SUBJ = sorted(subj_sess.keys())

VALID_SUBJ = sorted(
    s for s in ALL_SUBJ
    if sum(len(v) for v in subj_sess[s].values()) >= N_LABELED + N_UNLABELED + N_TEST
)
log.info(f'Soggetti validi (≥{N_LABELED+N_UNLABELED+N_TEST} trial): {len(VALID_SUBJ)}')


## §2 — Feature Extraction: 12 Temporal Features da Gamma (Table 2 del paper)

Identiche a Li et al. Table 2:
T1 centroid temporale · T2 mean abs diff · T3 mean diff · T4 median abs diff ·
T5 median diff · T6 signal traveled distance · T7 sum abs diff · T8 slope ·
T9 area under curve · T10 peak-to-peak · T11 Shannon entropy · T12 neighborhood peaks


In [ ]:
_b_g, _a_g = scipy_signal.butter(4, [GAMMA_LOW, GAMMA_HIGH], btype='bandpass', fs=SFREQ)

def _entropy(sig, n_bins=16):
    p, _ = np.histogram(sig, bins=n_bins, density=True)
    p = p[p > 0]; p = p / p.sum()
    return -np.sum(p * np.log2(p + 1e-12))

def temporal_features_12(sig):
    """sig: (T,) gamma-filtered signal for one channel → (12,) float64"""
    T  = len(sig)
    t  = np.arange(T, dtype=np.float64)
    d  = np.diff(sig)
    ad = np.abs(d)
    T1  = np.dot(t, np.abs(sig)) / (np.abs(sig).sum() + 1e-12)
    T2  = ad.mean()
    T3  = d.mean()
    T4  = np.median(ad)
    T5  = np.median(d)
    T6  = np.sqrt(1.0 + d**2).sum()
    T7  = ad.sum()
    T8  = np.polyfit(t, sig, 1)[0]
    T9  = np.abs(sig).sum() / SFREQ
    T10 = sig.max() - sig.min()
    T11 = _entropy(sig)
    T12 = float(((sig[1:-1] > sig[:-2]) & (sig[1:-1] > sig[2:])).sum())
    return np.array([T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T11,T12], dtype=np.float64)

def extract_trial_12feat(x_np):
    """x_np: (61, 384) → (61, 12) gamma temporal features"""
    xf = scipy_signal.filtfilt(_b_g, _a_g, x_np, axis=1)
    return np.stack([temporal_features_12(xf[c]) for c in range(N_CHANNELS)])  # (61,12)


# Cache v4: carica TUTTE le sessioni disponibili per ogni soggetto valido
FEAT_CACHE = FIG_DIR / 'eeg15v4_raw12.npz'
if FEAT_CACHE.exists():
    log.info(f'Cache trovata: {FEAT_CACHE}')
    _c = np.load(FEAT_CACHE)
    RAW12 = _c['features'];  LABELS = _c['labels'].astype(int)
    SUBJS = _c['subjects'].astype(int)
else:
    log.info('Estrazione feature gamma — tutte le sessioni (~10-15 min)...')
    fl, ll, sl = [], [], []
    for sid in tqdm(VALID_SUBJ, desc='Soggetti'):
        for ses_trials in subj_sess[sid].values():
            for p in ses_trials:
                try:
                    d = torch.load(p, weights_only=False)
                    x = d['x'].float().numpy()
                    yw = int(d['y'].squeeze() if isinstance(d['y'], torch.Tensor) else d['y'])
                    c  = label2cluster.get(yw)
                    if c is None: continue
                    fl.append(extract_trial_12feat(x)); ll.append(c); sl.append(sid)
                except: pass
    RAW12  = np.stack(fl).astype(np.float64)
    LABELS = np.array(ll, dtype=np.int32)
    SUBJS  = np.array(sl, dtype=np.int32)
    np.savez(FEAT_CACHE, features=RAW12, labels=LABELS, subjects=SUBJS)
    log.info(f'Salvato: {RAW12.shape}')

log.info(f'Totale trial: {len(RAW12)},  shape per trial: {RAW12.shape[1:]}')


## §3 — Selezione Top-2 Feature per PCC (Li et al. §4.1)

Calcola PCC assoluto medio tra ogni feature temporale e le label (one-vs-rest, media su canali e classi).
Seleziona le N_SELECT_FEAT=2 feature con |PCC| più alto → feature finale = 2×61 = 122D.


In [ ]:
FEAT_NAMES = ['T1_centroid','T2_mean_adiff','T3_mean_diff','T4_med_adiff','T5_med_diff',
              'T6_travel','T7_sum_adiff','T8_slope','T9_area','T10_ptp',
              'T11_entropy','T12_peaks']

def select_top_features(X12_l, Y_l, n_select=N_SELECT_FEAT):
    """
    Seleziona le top-n_select feature temporali per |PCC| medio su labeled set.
    X12_l: (n_l, 61, 12),  Y_l: (n_l,) → array di indici (n_select,)
    Identico a Li et al. §4.1: PCC per-canale one-vs-rest, media su canali e classi.
    """
    pcc_scores = np.zeros(12)
    for f in range(12):
        rs = []
        for ch in range(N_CHANNELS):
            v = X12_l[:, ch, f]
            for cls in range(N_CLASSES):
                yb = (Y_l == cls).astype(np.float64)
                if yb.std() < 1e-8: continue
                r, _ = pearsonr(v, yb)
                if not np.isnan(r): rs.append(abs(r))
        pcc_scores[f] = np.mean(rs) if rs else 0.0
    return np.argsort(pcc_scores)[-n_select:]

log.info('Funzione select_top_features caricata — PCC calcolato per-soggetto in §5')


## §4 — DHSLP: Algorithm 1 (implementazione esatta)

Tutte le formule dal paper:
- **Laplaciano** (eq. 3): L = diag(u) − Dᵥ⁻¹/² diag(u) H diag(w) Dₑ⁻¹ Hᵀ diag(u) Dᵥ⁻¹/²
- **Fu** (eq. 14): Fu = −Luu⁻¹ Lul Yl
- **Proj** (eq. 17-18): f eigenvettori minimi di αXᵀLX
- **H, u** (eq. 19-20): distanza-based nel sottospazio proiettato
- **w** (eq. 23-24): proiezione sul simplesso di (p+αq)/(2β)


In [ ]:
# ── Hyperedge construction (distance-based, eq.4) ────────────────────────────

def pairwise_sq_dist(X):
    """(n,d) → (n,n) squared Euclidean distances (numerically stable)."""
    sq = (X**2).sum(axis=1)
    return np.maximum(sq[:, None] + sq[None, :] - 2.0 * X @ X.T, 0.0)

def build_H_and_u(X, xi):
    """
    X: (n, d)
    Returns:
      H: (n, n) float64 incidence (row i = hyperedge eᵢ)
      u: (n,)   vertex weights (diag of U)
      D_sq: (n,n) squared distances (for reuse)
    """
    D_sq  = pairwise_sq_dist(X)
    D     = np.sqrt(D_sq)                         # (n,n)
    d_avg = D.mean(axis=1)                        # d̄ᵢ per ogni vertice
    thresh = xi * d_avg                           # threshold per ogni iperedge
    H = (D <= thresh[:, None]).astype(np.float64) # H[i,j]=1 se vj ∈ eᵢ
    # Vertex weights: u(vᵢ) = d̄ᵢ / Σd̄ⱼ  (eq.20)
    u = d_avg / (d_avg.sum() + 1e-12)
    return H, u, D_sq


# ── Laplaciano (eq.3) ─────────────────────────────────────────────────────────

def compute_L_and_factors(H, w, u):
    """
    H: (n,n)  w: (n,) hyperedge weights  u: (n,) vertex weights
    Returns L (n,n) and helper vectors dv_invsqrt, de_inv.
    """
    dv     = H @ w                                          # Dᵥ(i) = Σₑ H(i,e)w(e)
    de     = H.T @ u                                        # Dₑ(e) = Σᵥ H(v,e)u(v)
    dv_inv_sqrt = 1.0 / np.sqrt(np.maximum(dv, 1e-12))
    de_inv      = 1.0 / np.maximum(de, 1e-12)

    # A[i,e] = dv_inv_sqrt[i]*u[i]*H[i,e]*w[e]*de_inv[e]
    A  = (dv_inv_sqrt * u)[:, None] * H * (w * de_inv)[None, :]  # (n,n)
    # B = A @ Hᵀ * u[j]*dv_inv_sqrt[j]
    B  = A @ H.T * (u * dv_inv_sqrt)[None, :]                    # (n,n)
    L  = np.diag(u) - B
    return L, dv_inv_sqrt, de_inv


# ── W update: proiezione sul simplesso (eq.23-24) ────────────────────────────

def project_simplex(v):
    """Proietta vettore v sul simplesso di probabilità {w: Σwᵢ=1, wᵢ≥0}."""
    n = len(v)
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho  = np.where(u * np.arange(1, n+1) > (cssv - 1.0))[0]
    if len(rho) == 0: return np.full(n, 1.0/n)
    theta = (cssv[rho[-1]] - 1.0) / (rho[-1] + 1.0)
    return np.maximum(v - theta, 0.0)


def update_w(H, w, u, dv_inv_sqrt, de_inv, F, X, Proj, alpha, beta):
    """
    Calcola p, q dai fattori del Laplaciano, poi proietta sul simplesso.
    L_fac[e,i] = de_inv[e] * H[i,e] * u[i] * dv_inv_sqrt[i]  (→ (n,n) matrix)
    p = row_norms²(L_fac @ F),  q = row_norms²(L_fac @ X @ Proj)
    w_new = project_simplex((p + alpha*q) / (2*beta))
    """
    L_fac = de_inv[:, None] * H.T * (u * dv_inv_sqrt)[None, :]   # (n,n)

    ZF  = L_fac @ F                # (n, c)
    ZXP = L_fac @ (X @ Proj)      # (n, f)

    p = (ZF**2).sum(axis=1)        # (n,)
    q = (ZXP**2).sum(axis=1)       # (n,)

    w_unc = (p + alpha * q) / (2.0 * beta + 1e-12)
    return project_simplex(w_unc)


# ── Algorithm 1 completo ──────────────────────────────────────────────────────

def dhslp(X_l, Y_l_oh, X_u, f_dim, xi, alpha, beta, max_iter=MAX_ITER):
    """
    X_l: (n_l, d) labeled  Y_l_oh: (n_l, c) one-hot  X_u: (n_u, d) unlabeled
    Returns: Fu (n_u, c) pseudo-label scores,  Proj (d, f) projection matrix
    """
    n_l, d = X_l.shape
    n_u    = X_u.shape[0]
    n      = n_l + n_u
    c      = Y_l_oh.shape[1]
    f_dim  = min(f_dim, d)

    X  = np.vstack([X_l, X_u])                 # (n, d)
    F  = np.zeros((n, c), dtype=np.float64)
    F[:n_l] = Y_l_oh

    # Inizializzazione
    H, u, _  = build_H_and_u(X, xi)
    w        = np.full(n, 1.0/n)               # pesi iperedge uniformi

    Proj = np.eye(d, f_dim, dtype=np.float64)  # init M = I ridotta

    for _ in range(max_iter):

        # ── Step 1: Laplaciano ─────────────────────────────────────────────
        L, dv_inv_sqrt, de_inv = compute_L_and_factors(H, w, u)

        # ── Step 2: Update Fu (eq.14) ──────────────────────────────────────
        Lul = L[n_l:, :n_l]
        Luu = L[n_l:, n_l:]
        try:
            Fu = -np.linalg.solve(Luu + 1e-8*np.eye(n_u), Lul @ Y_l_oh)
        except np.linalg.LinAlgError:
            Fu = np.zeros((n_u, c))
        F[n_l:] = Fu

        # ── Step 3: Update Proj (eq.17-18) ────────────────────────────────
        XLX = alpha * (X.T @ L @ X)             # (d, d)
        XLX = (XLX + XLX.T) * 0.5              # symmetrize: evita errori in eigh
        try:
            _, vecs = eigh(XLX, check_finite=False)   # ascending eigenvalues
            Proj = vecs[:, :f_dim]               # f smallest eigenvectors
        except Exception:
            pass

        # ── Step 4: Update H, u nel sottospazio proiettato (eq.19-20) ─────
        X_proj       = X @ Proj                  # (n, f)
        H, u, _      = build_H_and_u(X_proj, xi)

        # ── Step 5: Update w (eq.23-24) ───────────────────────────────────
        L2, dv2, de2 = compute_L_and_factors(H, w, u)
        w = update_w(H, w, u, dv2, de2, F, X, Proj, alpha, beta)

    return Fu, Proj


log.info('DHSLP (Algorithm 1) caricato — pronto per il training')


## §5 — Loop Per-Soggetto + Grid Search (split identico al paper §4.1)

Per ogni soggetto:
1. **Split random** (seed=42): 300 labeled / 50 unlabeled / 50 test da tutte le sessioni miste
2. **PCC per-soggetto** su labeled set → top-2 feature → feature finale 122D
3. Normalizzazione StandardScaler (fit su X_l)
4. Grid search (ξ, α, β, f) → val bAcc su Fu (argmax)
5. Best config → test bAcc con **1-NN nel sottospazio Proj**
6. **Resume**: checkpoint salvato dopo ogni soggetto — ri-eseguire riprende da dove si era fermato

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

_ckpt_file = CKPT_DIR / 'eeg15v4_results.npy'
if _ckpt_file.exists():
    RESULTS = np.load(_ckpt_file, allow_pickle=True).item()
    log.info(f'Resume da checkpoint: {len(RESULTS)} soggetti già completati')
else:
    RESULTS = {}

for sid in tqdm(VALID_SUBJ, desc='Soggetti DHSLP'):
    if sid in RESULTS:
        continue

    mask    = (SUBJS == sid)
    X12_all = RAW12[mask]
    Y_all   = LABELS[mask]
    N_all   = len(Y_all)

    if N_all < N_LABELED + N_UNLABELED + N_TEST:
        log.warning(f'P{sid:03d}: solo {N_all} trial — skip')
        continue

    # Split stratificato per classe identico al paper §4.1
    # Paper: 60/classe × 5cl = 300 labeled, 10/classe × 5cl = 50 unlab, 10/classe × 5cl = 50 test
    # Adattato: 75/classe × 4cl = 300 labeled, ~12-13/classe × 4cl ≈ 50 unlab/test
    sss_te = StratifiedShuffleSplit(n_splits=1, test_size=N_TEST,
                                    random_state=RANDOM_SEED)
    idx_rest, idx_te = next(sss_te.split(X12_all, Y_all))

    sss_u = StratifiedShuffleSplit(n_splits=1, test_size=N_UNLABELED,
                                   random_state=RANDOM_SEED)
    idx_l_local, idx_u_local = next(sss_u.split(X12_all[idx_rest], Y_all[idx_rest]))
    idx_l = idx_rest[idx_l_local]
    idx_u = idx_rest[idx_u_local]

    X12_l  = X12_all[idx_l];  Y_l  = Y_all[idx_l]
    X12_u  = X12_all[idx_u];  Y_u  = Y_all[idx_u]
    X12_te = X12_all[idx_te]; Y_te = Y_all[idx_te]

    # PCC per-soggetto su labeled set
    top_idx = select_top_features(X12_l, Y_l)

    X_l  = X12_l[:,  :, top_idx].reshape(len(idx_l),  -1)
    X_u  = X12_u[:,  :, top_idx].reshape(len(idx_u),  -1)
    X_te = X12_te[:, :, top_idx].reshape(len(idx_te), -1)

    mu = X_l.mean(axis=0); sg = X_l.std(axis=0) + 1e-8
    Xl  = (X_l  - mu) / sg
    Xu  = (X_u  - mu) / sg
    Xte = (X_te - mu) / sg

    Y_l_oh = np.eye(N_CLASSES)[Y_l].astype(np.float64)

    best_val, best_cfg, best_Proj = 0.0, None, None
    _logged_err = False

    for xi in XI_LIST:
        for alpha in ALPHA_LIST:
            for beta in BETA_LIST:
                for f_dim in F_DIM_LIST:
                    try:
                        Fu, Proj = dhslp(Xl, Y_l_oh, Xu, f_dim, xi, alpha, beta)
                        vb = balanced_accuracy_score(Y_u, Fu.argmax(axis=1))
                        if vb > best_val:
                            best_val, best_cfg, best_Proj = vb, (xi, alpha, beta, f_dim), Proj
                    except Exception as e:
                        if not _logged_err:
                            log.debug(f'P{sid:03d} cfg ({xi},{alpha},{beta},{f_dim}): {e}')
                            _logged_err = True

    if best_cfg is None:
        log.warning(f'P{sid:03d}: nessuna config valida')
        RESULTS[sid] = dict(val_bacc=0.0, test_bacc=0.0, cfg=None, preds=None, gt=Y_te)
        np.save(_ckpt_file, RESULTS)
        continue

    knn = KNeighborsClassifier(n_neighbors=1, metric='euclidean')
    knn.fit(Xl @ best_Proj, Y_l)
    preds_te  = knn.predict(Xte @ best_Proj)
    test_bacc = balanced_accuracy_score(Y_te, preds_te)

    RESULTS[sid] = dict(val_bacc=best_val, test_bacc=test_bacc,
                        cfg=best_cfg, preds=preds_te, gt=Y_te,
                        top_feat_idx=top_idx.tolist())
    np.save(_ckpt_file, RESULTS)

    xi, al, be, fd = best_cfg
    log.info(f'P{sid:03d}: val={best_val:.4f} test={test_bacc:.4f}  '
             f'xi={xi} a={al} b={be} f={fd}  '
             f'n_l={len(idx_l)} n_u={len(idx_u)} n_te={len(idx_te)}')

chance = 1 / N_CLASSES
valid_res = {k: v for k, v in RESULTS.items() if v['cfg'] is not None}
if valid_res:
    tbs = [r['test_bacc'] for r in valid_res.values()]
    print(f'\n── DHSLP (Li et al. 2025) — {len(valid_res)}/{len(VALID_SUBJ)} soggetti ──')
    print(f'Mean test bAcc = {np.mean(tbs):.4f} ± {np.std(tbs):.4f}')
    print(f'% sopra chance = {(np.array(tbs) > chance).mean()*100:.1f}%  (chance={chance:.2%})')


## §6 — Risultati Finali + W&B

In [ ]:
# Ricarica da file se RESULTS vuoto
if not RESULTS:
    loaded = np.load(CKPT_DIR/'eeg15_results.npy', allow_pickle=True).item()
    RESULTS.update(loaded)

if not RESULTS:
    print('[INFO] Nessun risultato — esegui §5 prima.')
else:
    chance = 1 / N_CLASSES
    rows = []
    for sid, res in RESULTS.items():
        xi,al,be,fd = res['cfg']
        rows.append({'Subject': f'P{sid:03d}',
                     'Val bAcc': round(res['val_bacc'],4),
                     'Test bAcc': round(res['test_bacc'],4),
                     'xi': xi, 'alpha': al, 'beta': be, 'f': fd})
    df = pd.DataFrame(rows).sort_values('Test bAcc', ascending=False).reset_index(drop=True)
    df.to_csv(FIG_DIR/'eeg15_subject_ranking.csv', index=False)
    print(df.to_string(index=False))

    # ── Bar chart ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('DHSLP — Li et al. 2025 (EEG_15)', fontsize=13, fontweight='bold')

    colors = ['#2C7BB6' if v > chance else '#D7191C' for v in df['Test bAcc']]
    axes[0].bar(df['Subject'], df['Test bAcc'], color=colors, alpha=0.9)
    axes[0].axhline(chance, color='k', linestyle='--', linewidth=1.5,
                    label=f'Chance ({chance:.0%})')
    axes[0].set_title(f'Test bAcc per soggetto (sess.5)\nMean={df["Test bAcc"].mean():.4f}')
    axes[0].set_ylabel('Balanced Accuracy'); axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].tick_params(axis='x', rotation=90, labelsize=6)

    # Confusion matrix soggetto migliore
    best_sid = df.iloc[0]['Subject']
    best_sid_int = int(best_sid[1:])
    if best_sid_int in RESULTS:
        cm_vals = confusion_matrix(RESULTS[best_sid_int]['gt'],
                                   RESULTS[best_sid_int]['preds'], normalize='true')
        im = axes[1].imshow(cm_vals, cmap='Blues', vmin=0, vmax=1)
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                axes[1].text(j, i, f'{cm_vals[i,j]:.2f}', ha='center', va='center',
                            fontsize=10, color='white' if cm_vals[i,j]>0.5 else 'black')
        LBL = ['CONCR','AZIONE','STATO','ASTRATTO']
        axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(LBL, rotation=30)
        axes[1].set_yticks(range(N_CLASSES)); axes[1].set_yticklabels(LBL)
        axes[1].set_title(f'Confusion Matrix — {best_sid} (best)\nbAcc={df.iloc[0]["Test bAcc"]:.4f}')
        fig.colorbar(im, ax=axes[1], fraction=0.046)

    plt.tight_layout()
    plt.savefig(FIG_DIR/'eeg15_results.png', dpi=150); plt.show()
    log.info(f'Salvato: {FIG_DIR}/eeg15_results.png')

    # ── W&B ────────────────────────────────────────────────────────────────
    wandb.login()
    run = wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name=f'eeg15_DHSLP_LiEtAl2025_{CLUSTER_SCHEME}',
        config=dict(
            notebook='EEG_15', model='DHSLP', paper='Li_et_al_2025',
            doi='10.1088/1741-2552/adeec8',
            n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
            n_channels=N_CHANNELS, gamma_low=GAMMA_LOW, gamma_high=GAMMA_HIGH,
            n_select_feat=N_SELECT_FEAT,
            top_features=[FEAT_NAMES[i] for i in TOP_IDX],
            xi_list=XI_LIST, alpha_list=ALPHA_LIST,
            beta_list=BETA_LIST, f_dim_list=F_DIM_LIST,
            max_iter=MAX_ITER, n_subjects=len(RESULTS),
            train_sessions=TRAIN_SESSIONS, val_sessions=VAL_SESSIONS,
            test_sessions=TEST_SESSIONS,
        ),
        reinit='finish_previous',
        settings=wandb.Settings(start_method='thread')
    )
    run.log({
        'test/mean_bacc':         float(df['Test bAcc'].mean()),
        'test/std_bacc':          float(df['Test bAcc'].std()),
        'test/pct_above_chance':  float((df['Test bAcc'] > chance).mean()),
        'val/mean_bacc':          float(df['Val bAcc'].mean()),
    })
    run.summary.update({'test_mean_bacc': float(df['Test bAcc'].mean()),
                        'n_subjects': len(RESULTS)})
    run.finish()
    log.info('W&B logging completato')
    print(f'\nTop-5 soggetti:\n{df.head(5).to_string(index=False)}')
